<a href="https://colab.research.google.com/github/shreeyashbaran/Info-5731/blob/main/In_Class_Exercise_5%266_Feature_Extraction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [1]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()   # choose product_reviews.txt
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving product_reviews.txt to product_reviews.txt
Uploaded file name: product_reviews.txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [4]:
# Write your Answer here (Q1)
import pandas as pd

df = pd.read_csv(filename, sep='\t')

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print(df.head(3))

Columns: ['id', 'text']
Shape: (10, 2)
   id                                               text
0   1  Love this blender! Smoothies are super creamy ...
1   2  Terrible quality... stopped working after 2 da...
2   3      Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [5]:
# Write your Answer here (Q2)
import re
import string

# 1) Number of words
df["word_count"] = df["text"].fillna("").apply(lambda x: len(str(x).split()))

# 2) Number of characters
df["char_count"] = df["text"].fillna("").apply(lambda x: len(str(x)))

# 3) Average word length (ignoring punctuation)
def avg_word_len(text):
    text = str(text)
    words = text.split()

    # remove punctuation from each word
    clean_words = [word.strip(string.punctuation) for word in words]

    # keep only non-empty words
    clean_words = [word for word in clean_words if word]

    if len(clean_words) == 0:
        return 0

    total_len = sum(len(word) for word in clean_words)
    return total_len / len(clean_words)

df["avg_word_len"] = df["text"].fillna("").apply(avg_word_len)

# 4) Number of exclamation marks
df["excl_count"] = df["text"].fillna("").apply(lambda x: str(x).count("!"))

# Display the DataFrame
print(df)

   id                                               text  word_count  \
0   1  Love this blender! Smoothies are super creamy ...           9   
1   2  Terrible quality... stopped working after 2 da...           7   
2   3      Good value for the price. Shipping was quick.           8   
3   4  Not as described. Missing parts and the box wa...          10   
4   5  Customer support helped me get a replacement. ...           8   
5   6  Battery life is amazing, but the screen is too...          10   
6   7  I want a refund. Charged twice and nobody repl...           9   
7   8           Setup was easy. Instructions were clear.           6   
8   9  The app keeps crashing. Very frustrating exper...           7   
9  10   Works fine, nothing special. Wouldn’t buy again.           7   

   char_count  avg_word_len  excl_count  
0          55      5.000000           1  
1          51      5.571429           3  
2          45      4.500000           0  
3          56      4.500000           0

## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [8]:
# # Write your Answer here (Q3)
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Replace missing values with empty strings
texts = df["text"].fillna("")

# Create Bag-of-Words model
vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(texts)

# 1) Vocabulary size
vocab_size = len(vectorizer.get_feature_names_out())
print("Vocabulary size:", vocab_size)

# 2) Top 10 words by total count across all documents
word_counts = X.sum(axis=0).A1
words = vectorizer.get_feature_names_out()

bow_df = pd.DataFrame({
    "word": words,
    "count": word_counts
})

top_10_words = bow_df.sort_values(by="count", ascending=False).head(10)

print("\nTop 10 words by total count across all documents:")
print(top_10_words.to_string(index=False))


Vocabulary size: 50

Top 10 words by total count across all documents:
    word  count
 amazing      1
     app      1
 battery      1
 blender      1
     box      1
     buy      1
 charged      1
   clear      1
crashing      1
  creamy      1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [7]:
# Write your Answer here (Q4)
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Fill missing values
texts = df["text"].fillna("")

# Create bigram CountVectorizer
bigram_vectorizer = CountVectorizer(stop_words="english", ngram_range=(2, 2))
X_bigram = bigram_vectorizer.fit_transform(texts)

# Get bigram names and their total counts
bigrams = bigram_vectorizer.get_feature_names_out()
bigram_counts = X_bigram.sum(axis=0).A1

# Create a DataFrame for easy sorting
bigram_df = pd.DataFrame({
    "bigram": bigrams,
    "count": bigram_counts
})

# Print top 5 bigrams
top_5_bigrams = bigram_df.sort_values(by="count", ascending=False).head(5)

print("Top 5 bigrams by total count:")
print(top_5_bigrams.to_string(index=False))

Top 5 bigrams by total count:
           bigram  count
   amazing screen      1
        app keeps      1
     battery life      1
blender smoothies      1
      box damaged      1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [9]:
# Write your Answer here  (Q5)
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# Fill missing values
texts = df["text"].fillna("")

# Create TF-IDF vectorizer with unigrams + bigrams
tfidf_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(texts)

# Get feature names
terms = tfidf_vectorizer.get_feature_names_out()

# Compute average TF-IDF score for each term across all documents
avg_tfidf_scores = X_tfidf.mean(axis=0).A1

# Create DataFrame for sorting
tfidf_df = pd.DataFrame({
    "term": terms,
    "avg_score": avg_tfidf_scores
})

# Print top 5 terms
top_5_terms = tfidf_df.sort_values(by="avg_score", ascending=False).head(5)

print("Top 5 terms by average TF-IDF score:")
print(top_5_terms.to_string(index=False))

Top 5 terms by average TF-IDF score:
              term  avg_score
              easy   0.037796
             clear   0.037796
             setup   0.037796
instructions clear   0.037796
      instructions   0.037796


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
